# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shah833/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content article, on one day. Time window: March 2026

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

###**Label**: trend_direction

###**Features**: avg_position, ctr, competition_level, content_type

### **Context**: client_id (grouping only), word_count

### **Excluded**: trend_pct (label source), client names/URLs (privacy)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [5]:
# Query 1 grain check (should return empty if correct)

q1 = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [content_hash_id, report_date, row_count]
Index: []


In [6]:
# Query 2 Row count and date span

q2 = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as earliest, MAX(report_date) as latest
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
""").df()
print(q2)

   total_rows   earliest     latest
0     9841378 2026-03-01 2026-03-31


In [9]:
# Query 3 Availability check

q3 = con.sql(f"""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03' AND ga4_data_available IS TRUE
""").df()
print(q3)

   available_rows
0          413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### This data can't tell us much about engagement for most of March, because only about 4% of rows (413,966 out of 9.8 million) actually have real GA4 data, the rest are missing it. So anything which is based on CTR, scroll rate, or engagement is only trustworthy for a small slice of the data in this dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.